# **Query Translation**

For an in-depth discussion on the rationale behind Query Translation, including its specific use cases, methodological approaches, and typological distinctions, refer to the detailed documentation provided [here](what_is_query_translation.md).


##  Types of **Query Translation**
- Multi Query Translation
- HyDE (Hypothetical Document Embeddings)
- RAG Fusion
- Step-Back Reformulation
- Decomposition-Based Reformulation

| Technique         | Goal                        | When to Use            | Advantage                    | Limitation                   |
| ----------------- | --------------------------- | ---------------------- | ---------------------------- | ---------------------------- |
| **Multi-Query**   | Expand coverage             | Ambiguous queries      | Improves recall              | Higher compute cost          |
| **HyDE**          | Semantic enrichment         | Short or vague queries | Contextually rich retrieval  | Requires powerful generator  |
| **RAG Fusion**    | Combine multiple retrievals | Knowledge-heavy tasks  | Balanced precision-recall    | Complexity in fusion logic   |
| **Step-Back**     | Broaden context             | Narrow or deep queries | Enables high-level reasoning | Risk of diluting specificity |
| **Decomposition** | Structured reasoning        | Multi-hop queries      | Supports stepwise inference  | Integration complexity       |

### Multi Query Translation

- Loading and chunking the Documents

In [1]:
# Import necessary class
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Create a text splitter with defined chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 50,    # Each chunk will contain around 10 tokens
    chunk_overlap = 3   # 3 tokens of overlap between consecutive chunks
)

def load_documents():
    """
    Loads a small, diverse set of documents to simulate a knowledge base
    for Retrieval-Augmented Generation (RAG) experiments.

    Each document represents a single knowledge unit that will later be
    chunked, embedded, and indexed for similarity-based retrieval.

    Returns
    -------
    documents : list of langchain_core.documents.Document
        A list containing 10 textual documents across multiple domains.
    """

    # Define a diverse mini knowledge base covering various topics
    documents = [
        Document(page_content="The Eiffel Tower is located in Paris, France, and stands at 324 meters tall."),
        Document(page_content="The Great Wall of China stretches over 13,000 miles and was built to prevent invasions."),
        Document(page_content="Machine learning enables systems to learn and improve automatically from experience."),
        Document(page_content="Python is widely used in data science due to its rich ecosystem of libraries like NumPy and Pandas."),
        Document(page_content="The Amazon rainforest produces 20% of the world’s oxygen and hosts vast biodiversity."),
        Document(page_content="Albert Einstein proposed the theory of relativity, which changed modern physics."),
        Document(page_content="Neural networks are computational models inspired by the human brain’s structure."),
        Document(page_content="The Sun is approximately 93 million miles away from Earth and is the center of our solar system."),
        Document(page_content="Retrieval-Augmented Generation (RAG) combines retrieval and generation to enhance factual accuracy."),
        Document(page_content="OpenAI developed GPT models that leverage massive datasets to achieve human-like text generation.")
    ]

    # Log summary information for verification
    print(f"Total documents loaded: {len(documents)}")
    print("Displaying sample documents:")
    for i, doc in enumerate(documents[:3]):
        print(f"Document {i+1}: {doc.page_content[:100]}...")

    return documents
    
documents = load_documents()

# Split the document into chunks
chunks = text_splitter.split_documents(documents)

# Display the created chunks
print(f"\nTotal chunks created: {len(chunks)}")
print("Displaying first 2 sample chunks:")
for i, chunk in enumerate(chunks[:2]):
    print(f"Chunk {i+1}: {chunk.page_content}")

Total documents loaded: 10
Displaying sample documents:
Document 1: The Eiffel Tower is located in Paris, France, and stands at 324 meters tall....
Document 2: The Great Wall of China stretches over 13,000 miles and was built to prevent invasions....
Document 3: Machine learning enables systems to learn and improve automatically from experience....

Total chunks created: 10
Displaying first 2 sample chunks:
Chunk 1: The Eiffel Tower is located in Paris, France, and stands at 324 meters tall.
Chunk 2: The Great Wall of China stretches over 13,000 miles and was built to prevent invasions.


- Indexing the Documents

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
model = SentenceTransformer("all-MiniLM-L6-v2")
# Initialize an empty vector store to hold embeddings for each chunk
vector_store = []

# Convert each text chunk into a dense vector representation (embedding)
for idx, chunk in enumerate(chunks):
    text_content = chunk.page_content
    embedding_vector = model.encode(text_content)   # Convert text → vector
    vector_store.append(embedding_vector)

# Display the first 5 dimensions of first two chunk embeddings (for inspection)
[vector_store[0][:5], vector_store[1][:5]]

[array([ 0.06086044,  0.06121252, -0.00085703, -0.03503641, -0.01374615],
       dtype=float32),
 array([ 0.01918813,  0.10575976,  0.02296083, -0.02900702,  0.03200015],
       dtype=float32)]

- Multi-Query Translation

In [3]:
# Function to calculate cosine similarity between two difference vectors
def calculate_cosine_similarity_score_vect(v1, v2):
    dot_product = np.dot(v1, v2)
    v1_magnitude = np.linalg.norm(v1)
    v2_magnitude = np.linalg.norm(v2)
    cosine_similarity = dot_product / (v1_magnitude * v2_magnitude)
    return cosine_similarity

def retrieve_most_similar_chunk(model, chunks, vector_store, query_text):
    """
    Given a query, finds the most semantically similar chunk.
    Steps:
    1. Encode the query text into a vector.
    2. Compute cosine similarity with every chunk vector.
    3. Return the chunk with the highest similarity score.
    """
    encoded_query = model.encode(query_text)

    similarity_scores = []
    indexed_scores = []

    # Calculate similarity for each stored embedding
    for idx, chunk_vector in enumerate(vector_store):
        score = calculate_cosine_similarity_score_vect(encoded_query, chunk_vector)
        similarity_scores.append(score)
        indexed_scores.append((score, idx))

    # Log all similarity values for reference
    print("Calculated cosine similarity scores for each chunk:")
    for score, idx in indexed_scores:
        print(f"Score = {score:.4f}  |  Chunk Index = {idx}")

    # Retrieve the chunk with the maximum similarity score
    best_match_index = np.argmax(similarity_scores)
    best_chunk = chunks[best_match_index].page_content
    return best_chunk

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAI

# --------------------------------------------------------------------------
# Multi-Query Generation Model
# --------------------------------------------------------------------------
# Purpose:
# Generate multiple semantically diverse reformulations of the user's query.
# Each reformulated query represents a different perspective to improve
# document retrieval recall and reduce semantic bias in vector similarity search.

# Define the prompt template for the LLM
prompt_template = ChatPromptTemplate.from_messages([
    ('system', 
    """
    You are an advanced language model. Your task is to generate five diverse reformulations 
    of the given user query to enhance document retrieval from a vector database.
    The objective is to capture multiple semantic perspectives of the original query,
    thereby mitigating the limitations of distance-based similarity search.
    Each reformulated query should be presented on a new line.
    """),
    ("user", "{user_query}")
])

# Initialize the generative LLM (Gemini 2.5-Flash)
llm = GoogleGenerativeAI(model="gemini-2.5-flash")

# Build the Multi-Query Generation Chain
# This chain takes the user query, sends it to the LLM, parses the output,
# and splits the generated reformulations line by line.
multi_query_chain = (prompt_template | llm | StrOutputParser() | (lambda x: x.split("\n")))

# Example user query
query = "What is Machine learning?"

# Generate multiple diverse reformulations for the given query
generated_queries = multi_query_chain.invoke(query)

print("Original Query:", query)
print("\nGenerated Queries by Multi-Query Generation Model:")
for idx, question in enumerate(generated_queries, start=1):
    print(f"Q.{idx}) {question}")

Original Query: What is Machine learning?

Generated Queries by Multi-Query Generation Model:
Q.1) Define Machine Learning.
Q.2) How do algorithms learn from data?
Q.3) What is the core concept of computers improving performance without explicit programming?
Q.4) Explain Machine Learning as a field within Artificial Intelligence.
Q.5) What constitutes a system that learns from experience?


- Reteriving the Documents for each and Every Question Generated by Multi-Query Generation Model

In [5]:
# --------------------------------------------------------------------------
# Reteriving the Documents for each and Every Question Generated by Multi-Query Generation Model
# --------------------------------------------------------------------------
# Purpose:
# For each reformulated query, retrieve the most semantically similar document
# chunks from the vector database. This step expands coverage across multiple
# formulations of the same information need.

retrieved_chunks = []  # To store retrieved chunks for each query

for question in generated_queries:
    print("\nProcessing Reformulated Query:", question)
    
    # Retrieve the top matching chunk for the current reformulated query
    retrieved_chunk = retrieve_most_similar_chunk(model, chunks, vector_store, question)
    
    retrieved_chunks.append(retrieved_chunk)

# Display the mapping between each reformulated query and its retrieved document chunk
print("\nRetrieved Chunks for Each Reformulated Query:")
for idx, question in enumerate(generated_queries):
    print(f"{idx+1}. {question} | Retrieved Chunk: {retrieved_chunks[idx]}")


Processing Reformulated Query: Define Machine Learning.
Calculated cosine similarity scores for each chunk:
Score = -0.0109  |  Chunk Index = 0
Score = -0.0186  |  Chunk Index = 1
Score = 0.6434  |  Chunk Index = 2
Score = 0.2072  |  Chunk Index = 3
Score = 0.0259  |  Chunk Index = 4
Score = 0.1605  |  Chunk Index = 5
Score = 0.3475  |  Chunk Index = 6
Score = 0.0995  |  Chunk Index = 7
Score = 0.1657  |  Chunk Index = 8
Score = 0.1000  |  Chunk Index = 9

Processing Reformulated Query: How do algorithms learn from data?
Calculated cosine similarity scores for each chunk:
Score = -0.0055  |  Chunk Index = 0
Score = 0.0044  |  Chunk Index = 1
Score = 0.4891  |  Chunk Index = 2
Score = 0.2455  |  Chunk Index = 3
Score = -0.0031  |  Chunk Index = 4
Score = 0.1291  |  Chunk Index = 5
Score = 0.3919  |  Chunk Index = 6
Score = 0.0408  |  Chunk Index = 7
Score = 0.1605  |  Chunk Index = 8
Score = 0.2233  |  Chunk Index = 9

Processing Reformulated Query: What is the core concept of computer

- Generation

In [6]:
# --------------------------------------------------------------------------
# Generation
# --------------------------------------------------------------------------
# Purpose:
# Use the retrieved context to synthesize a factual, concise, and research-oriented
# response. The LLM is instructed to maintain academic tone, factual accuracy,
# and avoid hallucination or unsupported reasoning.

agent_prompt = """
    You are an intelligent research assistant capable of grounded reasoning.
    Your task is to generate a concise, factual, and contextually accurate answer
    based on the retrieved context provided below.
    
    -------------------------------
    Context:
    {retrieved_chunk}
    -------------------------------
    
    Question:
    {user_query}

    Instructions:
    1. Use the provided retrieved context to answer the user query.
    2. If the context does not contain enough information, clearly state that.
    3. Avoid hallucination or assumptions beyond the provided context.
    4. Respond in clear, structured language suitable for a research explanation.
    5. Do not format the answer in markdown — respond in plain text only.
    6. Summarize the relevant information from the retrieved context before answering.
    7. Enhance the clarity, coherence, and academic tone of the response to ensure it reads like a refined summary, not raw text output.
    """

# Create the prompt and response generation chain
prompt = ChatPromptTemplate.from_messages([
    ("system", agent_prompt),
    ("user", "{user_query}")
])

# Build the reasoning-generation chain
chain = (prompt | llm | StrOutputParser())

# Invoke the chain to generate a final answer using the retrieved context
response = chain.invoke({
    "retrieved_chunk": retrieved_chunks,
    "user_query": query
})

print("\nGenerated Response:\n")
print(response)


Generated Response:

Machine learning is a methodology through which systems acquire the capacity to learn and enhance their performance autonomously based on accumulated experience.
